# Data Mining - Project 3


## Problem 1 - Data Set familiarization

In [ ]:
!pip install ucimlrepo
!pip install scikit-learn
!pip install matplotlib
!pip install numpy
!pip install pandas

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN


In [15]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition = fetch_ucirepo(id=544) 
  
# data (as pandas dataframes) 
X = estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.data.features 
y = estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.data.targets.squeeze() 
  
# metadata 
print(estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.metadata) 
  
# variable information 
print(estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.variables) 


{'uci_id': 544, 'name': 'Estimation of Obesity Levels Based On Eating Habits and Physical Condition ', 'repository_url': 'https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition', 'data_url': 'https://archive.ics.uci.edu/static/public/544/data.csv', 'abstract': 'This dataset include data for the estimation of obesity levels in individuals from the countries of Mexico, Peru and Colombia, based on their eating habits and physical condition. ', 'area': 'Health and Medicine', 'tasks': ['Classification', 'Regression', 'Clustering'], 'characteristics': ['Multivariate'], 'num_instances': 2111, 'num_features': 16, 'feature_types': ['Integer'], 'demographics': ['Gender', 'Age'], 'target_col': ['NObeyesdad'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2019, 'last_updated': 'Tue Sep 10 2024', 'dataset_doi': '10.24432/C5H31Z', 'creators': [], 'intro_paper': {'ID': 358, 'type': 

### 1.1 Why are you interested in this dataset?
I am interested in this dataset because I am personally curious about how obesity levels are correlated or not correlated to a persons eating a physical conditions. 

### 1.2 How many numerical attributes and categorical attributes are there in the dataset?
There are 8 numerical attributes and 9| categorical attributes. 

### 1.3 Are there any missing values in numerical attributes?
No there are no missing values.

### 1.4.1 Why do you think clusters will be present in this dataset?
I think there will be clusters present because a persons weight should be correlated to their eating habits and their physical conditions. 

### 1.4.2 Why might finding clusters in this dataset be helpful?
Doctors may be able to use this data to show how a persons current eating habits and physical conditions could predict their obesity level. 

### 1.4.3 How many clusters do you expect to see in this dataset?
I think there is going to be around 2 to 5 clusters. Mainly becuase a society may be more prone to one style of eating habits and physical conditions so it could be a 50/50 or there would be more. 

### 1.4.4 Do you expect to see clusters of similar size? Why or why not?
No, I imagine clusters will vary and create a linear distribution.

## Problem 2 - Clustering Function Implementations

The functions below implement `k`-means, DBSCAN, and clustering precision without calling existing clustering algorithm implementations. The last code cell contains small test cases for correctness.


In [2]:
from collections import deque

def k_means_clustering(data, k, epsilon, max_iter=300, random_state=42, initial_centroids=None):
    """Run k-means clustering using Euclidean distance.

    Returns a dictionary with cluster centroids, point labels, cluster membership,
    and the final objective value (sum of squared distances).
    """
    data = np.asarray(data, dtype=float)
    if data.ndim != 2:
        raise ValueError("data must be a 2D numeric matrix")
    n_samples, n_features = data.shape
    if not 1 <= k <= n_samples:
        raise ValueError("k must be between 1 and the number of samples")
    if epsilon < 0:
        raise ValueError("epsilon must be non-negative")

    if initial_centroids is not None:
        centroids = np.asarray(initial_centroids, dtype=float).copy()
        if centroids.shape != (k, n_features):
            raise ValueError("initial_centroids must have shape (k, n_features)")
    else:
        rng = np.random.default_rng(random_state)
        initial_indices = rng.choice(n_samples, size=k, replace=False)
        centroids = data[initial_indices].copy()

    for _ in range(max_iter):
        distances = np.sum((data[:, None, :] - centroids[None, :, :]) ** 2, axis=2)
        labels = np.argmin(distances, axis=1)
        new_centroids = centroids.copy()

        for cluster_index in range(k):
            cluster_points = data[labels == cluster_index]
            if len(cluster_points) > 0:
                new_centroids[cluster_index] = cluster_points.mean(axis=0)

        centroid_shift = np.max(np.linalg.norm(new_centroids - centroids, axis=1))
        centroids = new_centroids
        if centroid_shift <= epsilon:
            break

    final_distances = np.sum((data[:, None, :] - centroids[None, :, :]) ** 2, axis=2)
    final_labels = np.argmin(final_distances, axis=1)
    clusters = [np.where(final_labels == cluster_index)[0].tolist() for cluster_index in range(k)]
    objective_value = float(final_distances[np.arange(n_samples), final_labels].sum())

    return {
        "centroids": centroids,
        "labels": final_labels,
        "clusters": clusters,
        "objective_value": objective_value,
    }


In [3]:
def dbscan_clustering(data, minpts, epsilon):
    """Run DBSCAN and label each point as core, border, or noise."""
    data = np.asarray(data, dtype=float)
    if data.ndim != 2:
        raise ValueError("data must be a 2D numeric matrix")
    if minpts < 1:
        raise ValueError("minpts must be at least 1")
    if epsilon < 0:
        raise ValueError("epsilon must be non-negative")

    n_samples = data.shape[0]
    distance_matrix = np.linalg.norm(data[:, None, :] - data[None, :, :], axis=2)
    neighborhoods = [np.where(distance_matrix[i] <= epsilon)[0].tolist() for i in range(n_samples)]
    is_core = np.array([len(neighbors) >= minpts for neighbors in neighborhoods], dtype=bool)

    UNASSIGNED = -99
    NOISE = -1
    labels = np.full(n_samples, UNASSIGNED, dtype=int)
    visited = np.zeros(n_samples, dtype=bool)
    cluster_id = 0

    for point_index in range(n_samples):
        if visited[point_index]:
            continue

        visited[point_index] = True
        if not is_core[point_index]:
            labels[point_index] = NOISE
            continue

        labels[point_index] = cluster_id
        queue = deque(neighborhoods[point_index])
        queued = set(neighborhoods[point_index])

        while queue:
            neighbor_index = queue.popleft()
            if not visited[neighbor_index]:
                visited[neighbor_index] = True
                if is_core[neighbor_index]:
                    for expanded_neighbor in neighborhoods[neighbor_index]:
                        if expanded_neighbor not in queued:
                            queue.append(expanded_neighbor)
                            queued.add(expanded_neighbor)

            if labels[neighbor_index] in (UNASSIGNED, NOISE):
                labels[neighbor_index] = cluster_id

        cluster_id += 1

    labels[labels == UNASSIGNED] = NOISE
    point_types = np.full(n_samples, "noise", dtype=object)
    point_types[labels != NOISE] = "border"
    point_types[(labels != NOISE) & is_core] = "core"
    clusters = [np.where(labels == current_cluster)[0].tolist() for current_cluster in range(cluster_id)]

    return {
        "labels": labels,
        "clusters": clusters,
        "point_types": point_types.tolist(),
        "core_mask": is_core,
    }


In [4]:
def clustering_precision(true_labels, predicted_labels):
    """Compute pairwise clustering precision while ignoring predicted noise points."""
    true_labels = np.asarray(true_labels)
    predicted_labels = np.asarray(predicted_labels)

    if true_labels.shape[0] != predicted_labels.shape[0]:
        raise ValueError("true_labels and predicted_labels must have the same length")

    true_positives = 0
    predicted_positive_pairs = 0

    for i in range(len(true_labels) - 1):
        same_predicted_cluster = (predicted_labels[i + 1:] == predicted_labels[i]) & (predicted_labels[i] != -1)
        predicted_positive_pairs += int(np.sum(same_predicted_cluster))
        true_positives += int(np.sum(same_predicted_cluster & (true_labels[i + 1:] == true_labels[i])))

    if predicted_positive_pairs == 0:
        return 0.0
    return true_positives / predicted_positive_pairs


### Problem 2 Test Cases
These tests use small synthetic datasets so the expected clustering behavior is easy to verify.


In [5]:
# k-means: verify clear two-cluster separation and tie-breaking to the lowest-index cluster.
kmeans_tie_data = np.array([[0.0], [2.0], [4.0]])
kmeans_tie_result = k_means_clustering(
    kmeans_tie_data,
    k=2,
    epsilon=1e-9,
    initial_centroids=np.array([[0.0], [4.0]])
)
assert kmeans_tie_result["labels"][1] == 0

kmeans_data = np.array([[0.0, 0.0], [0.0, 1.0], [9.0, 9.0], [9.0, 10.0]])
kmeans_result = k_means_clustering(
    kmeans_data,
    k=2,
    epsilon=1e-9,
    initial_centroids=np.array([[0.0, 0.0], [9.0, 10.0]])
)
kmeans_cluster_sets = {frozenset(cluster) for cluster in kmeans_result["clusters"]}
assert kmeans_cluster_sets == {frozenset({0, 1}), frozenset({2, 3})}

# DBSCAN: verify border/core/noise labeling on a simple 1D dataset.
dbscan_data = np.array([[0.0], [0.1], [0.2], [0.3], [1.0]])
dbscan_result = dbscan_clustering(dbscan_data, minpts=3, epsilon=0.11)
assert len(dbscan_result["clusters"]) == 1
assert set(dbscan_result["clusters"][0]) == {0, 1, 2, 3}
assert dbscan_result["point_types"] == ["border", "core", "core", "border", "noise"]
assert dbscan_result["labels"][-1] == -1

# Clustering precision: perfect clustering should have precision 1.0, and mixed clusters should be lower.
assert abs(clustering_precision([0, 0, 1, 1], [1, 1, 0, 0]) - 1.0) < 1e-12
assert abs(clustering_precision([0, 0, 1, 1], [0, 0, 0, 1]) - (1 / 3)) < 1e-12

print("k-means test clusters:", kmeans_result["clusters"])
print("k-means objective value:", round(kmeans_result["objective_value"], 4))
print("DBSCAN point types:", dbscan_result["point_types"])
print("DBSCAN cluster labels:", dbscan_result["labels"].tolist())
print("All Problem 2 tests passed.")


k-means test clusters: [[0, 1], [2, 3]]
k-means objective value: 1.0
DBSCAN point types: ['border', 'core', 'core', 'border', 'noise']
DBSCAN cluster labels: [0, 0, 0, 0, -1]
All Problem 2 tests passed.


# Part 3 - Analyze your data


### 3.1 Use sklearn’s PCA library to linearly transform the dataset into two dimensions.
The scatter plot below uses only the numerical attributes after standardization. There appears to be visible structure with overlapping groups rather than perfectly separated clusters. A reasonable visual estimate is that there are about **5 to 7 clusters**, though they are not sharply separated in two dimensions.


In [ ]:
# Select only numerical attributes for the clustering analysis required by the project
numerical_columns = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
X_num = X[numerical_columns].copy()

# Standardize before PCA and clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num)

# PCA to two dimensions for visualization
pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca_2[:, 0], X_pca_2[:, 1], alpha=0.6, s=18)
plt.xlabel('1st Principal Component')
plt.ylabel('2nd Principal Component')
plt.title('Problem 3.1: PCA Projection to Two Dimensions')
plt.grid(True, alpha=0.3)
plt.show()

print('Explained variance ratio for first two PCs:', pca_2.explained_variance_ratio_)
print('Combined variance captured by first two PCs:', pca_2.explained_variance_ratio_.sum())


### 3.2 Use sklearn’s PCA library without inputting the number of components.
The cumulative variance plot is used to choose a reduced number of dimensions. A good choice here is **7 principal components**, because they capture about **95.46%** of the total variance while still reducing dimensionality.


In [ ]:
pca_full = PCA()
pca_full.fit(X_scaled)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)
components = np.arange(1, len(explained_variance) + 1)

plt.figure(figsize=(8, 6))
plt.plot(components, cumulative_variance, marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Fraction of Variance Captured')
plt.title('Problem 3.2: Cumulative Explained Variance')
plt.xticks(components)
plt.grid(True, alpha=0.3)
plt.show()

variance_table = pd.DataFrame({
    'Principal Components': components,
    'Explained Variance Ratio': explained_variance,
    'Cumulative Variance': cumulative_variance
})
print(variance_table)

chosen_r = 7
print(f'Chosen number of components: {chosen_r}')
print(f'Fraction of variance captured by first {chosen_r} components: {cumulative_variance[chosen_r - 1]:.6f}')

pca_reduced = PCA(n_components=chosen_r)
X_reduced = pca_reduced.fit_transform(X_scaled)


### 3.3 For both the original and reduced datasets, run k-means for a range of k values.
The elbow plots below use **8 different k values**. In both cases, the objective value drops quickly and then starts to level off around **k = 6 or 7**, which makes those values reasonable choices.


In [ ]:
k_values = list(range(2, 10))
objective_original = []
objective_reduced = []

for k in k_values:
    km_original = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km_original.fit(X_scaled)
    objective_original.append(km_original.inertia_)

    km_reduced = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km_reduced.fit(X_reduced)
    objective_reduced.append(km_reduced.inertia_)

plt.figure(figsize=(8, 6))
plt.plot(k_values, objective_original, marker='o')
plt.xlabel('k')
plt.ylabel('Objective Function Value (Inertia)')
plt.title('Problem 3.3: k-means on Original Standardized Data')
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 6))
plt.plot(k_values, objective_reduced, marker='o')
plt.xlabel('k')
plt.ylabel('Objective Function Value (Inertia)')
plt.title('Problem 3.3: k-means on PCA-Reduced Data')
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.show()

print('Original-data objective values:')
print(pd.DataFrame({'k': k_values, 'Objective Value': objective_original}))

print('\nReduced-data objective values:')
print(pd.DataFrame({'k': k_values, 'Objective Value': objective_reduced}))


### 3.4 For both the original and reduced datasets, run DBSCAN over a range of epsilon and minpts values.
Below, epsilon is varied while keeping `minpts = 5`, and then `minpts` is varied while keeping `epsilon = 1.25`. This gives at least 6 values for each parameter as required. The results show that DBSCAN is quite sensitive on this dataset, which suggests overlapping density regions rather than one obvious cluster scale.


In [ ]:
def number_of_clusters(labels):
    labels = np.asarray(labels)
    return len(set(labels)) - (1 if -1 in labels else 0)

epsilon_values = [0.50, 0.75, 1.00, 1.25, 1.50, 1.75]
minpts_values = [3, 4, 5, 6, 7, 8]

fixed_minpts = 5
fixed_epsilon = 1.25

eps_results = []
for eps in epsilon_values:
    labels_original = DBSCAN(eps=eps, min_samples=fixed_minpts).fit_predict(X_scaled)
    labels_reduced = DBSCAN(eps=eps, min_samples=fixed_minpts).fit_predict(X_reduced)
    eps_results.append({
        'epsilon': eps,
        'minpts': fixed_minpts,
        'clusters_original': number_of_clusters(labels_original),
        'clusters_reduced': number_of_clusters(labels_reduced)
    })

minpts_results = []
for m in minpts_values:
    labels_original = DBSCAN(eps=fixed_epsilon, min_samples=m).fit_predict(X_scaled)
    labels_reduced = DBSCAN(eps=fixed_epsilon, min_samples=m).fit_predict(X_reduced)
    minpts_results.append({
        'epsilon': fixed_epsilon,
        'minpts': m,
        'clusters_original': number_of_clusters(labels_original),
        'clusters_reduced': number_of_clusters(labels_reduced)
    })

eps_df = pd.DataFrame(eps_results)
minpts_df = pd.DataFrame(minpts_results)

print('DBSCAN results while varying epsilon (minpts fixed at 5):')
print(eps_df)

print('\nDBSCAN results while varying minpts (epsilon fixed at 1.25):')
print(minpts_df)


### 3.5 Extra Credit: Plot clustering precision.
The code below computes clustering precision for the same k, epsilon, and minpts values on both the original and reduced datasets, producing the required 6 plots.


In [ ]:
# Factorize class labels for extra-credit precision comparisons
true_labels = pd.factorize(y)[0]

precision_k_original = []
precision_k_reduced = []

for k in k_values:
    pred_original = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300).fit_predict(X_scaled)
    pred_reduced = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300).fit_predict(X_reduced)
    precision_k_original.append(clustering_precision(true_labels, pred_original))
    precision_k_reduced.append(clustering_precision(true_labels, pred_reduced))

plt.figure(figsize=(8, 6))
plt.plot(k_values, precision_k_original, marker='o')
plt.xlabel('k')
plt.ylabel('Clustering Precision')
plt.title('Problem 3.5: Precision vs k (Original Data)')
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 6))
plt.plot(k_values, precision_k_reduced, marker='o')
plt.xlabel('k')
plt.ylabel('Clustering Precision')
plt.title('Problem 3.5: Precision vs k (Reduced Data)')
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.show()

precision_eps_original = []
precision_eps_reduced = []

for eps in epsilon_values:
    pred_original = DBSCAN(eps=eps, min_samples=fixed_minpts).fit_predict(X_scaled)
    pred_reduced = DBSCAN(eps=eps, min_samples=fixed_minpts).fit_predict(X_reduced)
    precision_eps_original.append(clustering_precision(true_labels, pred_original))
    precision_eps_reduced.append(clustering_precision(true_labels, pred_reduced))

plt.figure(figsize=(8, 6))
plt.plot(epsilon_values, precision_eps_original, marker='o')
plt.xlabel('epsilon')
plt.ylabel('Clustering Precision')
plt.title('Problem 3.5: Precision vs epsilon (Original Data)')
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 6))
plt.plot(epsilon_values, precision_eps_reduced, marker='o')
plt.xlabel('epsilon')
plt.ylabel('Clustering Precision')
plt.title('Problem 3.5: Precision vs epsilon (Reduced Data)')
plt.grid(True, alpha=0.3)
plt.show()

precision_minpts_original = []
precision_minpts_reduced = []

for m in minpts_values:
    pred_original = DBSCAN(eps=fixed_epsilon, min_samples=m).fit_predict(X_scaled)
    pred_reduced = DBSCAN(eps=fixed_epsilon, min_samples=m).fit_predict(X_reduced)
    precision_minpts_original.append(clustering_precision(true_labels, pred_original))
    precision_minpts_reduced.append(clustering_precision(true_labels, pred_reduced))

plt.figure(figsize=(8, 6))
plt.plot(minpts_values, precision_minpts_original, marker='o')
plt.xlabel('minpts')
plt.ylabel('Clustering Precision')
plt.title('Problem 3.5: Precision vs minpts (Original Data)')
plt.xticks(minpts_values)
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 6))
plt.plot(minpts_values, precision_minpts_reduced, marker='o')
plt.xlabel('minpts')
plt.ylabel('Clustering Precision')
plt.title('Problem 3.5: Precision vs minpts (Reduced Data)')
plt.xticks(minpts_values)
plt.grid(True, alpha=0.3)
plt.show()

print('Precision for k values (original data):', precision_k_original)
print('Precision for k values (reduced data):', precision_k_reduced)
print('Precision for epsilon values (original data):', precision_eps_original)
print('Precision for epsilon values (reduced data):', precision_eps_reduced)
print('Precision for minpts values (original data):', precision_minpts_original)
print('Precision for minpts values (reduced data):', precision_minpts_reduced)


### Problem 3 - Summary
- **3.1:** The PCA scatter plot shows visible structure with overlapping groups; about 5 to 7 clusters seem plausible.
- **3.2:** The cumulative variance reaches about 95.46% at 7 principal components, so 7 is a strong reduced-dimension choice.
- **3.3:** The k-means elbow appears around k = 6 or 7 for both the original and reduced datasets.
- **3.4:** DBSCAN is sensitive to epsilon and minpts on this dataset, which suggests overlapping density structure instead of one single natural scale.
- **3.5 Extra Credit:** Precision plots are produced for all required k, epsilon, and minpts settings on both original and reduced datasets.
